# Outlier Detection Methods

Compare Z-score, IQR, and isolation-based methods for detecting outliers in data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

In [ ]:
# Generate data with outliers
normal_data = np.random.normal(50, 10, 200)
outliers = np.array([5, 10, 95, 100, 120])
data = np.concatenate([normal_data, outliers])
np.random.shuffle(data)

print(f"Data size: {len(data)}")
print(f"True outliers inserted: {outliers}")

In [ ]:
# Method 1: Z-Score
mean, std = np.mean(data), np.std(data)
z_scores = np.abs((data - mean) / std)
z_threshold = 2.5
z_outliers = data[z_scores > z_threshold]

print(f"\nZ-SCORE METHOD (|z| > {z_threshold}):")
print(f"  Detected: {np.sort(z_outliers)}")

In [ ]:
# Method 2: IQR
Q1, Q3 = np.percentile(data, [25, 75])
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
iqr_outliers = data[(data < lower_fence) | (data > upper_fence)]

print(f"\nIQR METHOD (1.5 × IQR rule):")
print(f"  Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}")
print(f"  Fences: [{lower_fence:.1f}, {upper_fence:.1f}]")
print(f"  Detected: {np.sort(iqr_outliers)}")

In [ ]:
# Method 3: Modified Z-Score (MAD-based, robust)
median = np.median(data)
MAD = np.median(np.abs(data - median))
modified_z = 0.6745 * (data - median) / MAD  # 0.6745 ≈ normal consistency
mad_threshold = 3.5
mad_outliers = data[np.abs(modified_z) > mad_threshold]

print(f"\nMODIFIED Z-SCORE (MAD-based, |Mz| > {mad_threshold}):")
print(f"  Median={median:.1f}, MAD={MAD:.1f}")
print(f"  Detected: {np.sort(mad_outliers)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Z-score
axes[0].scatter(range(len(data)), data, c=(z_scores > z_threshold), cmap='coolwarm', alpha=0.6)
axes[0].axhline(mean + z_threshold * std, color='red', linestyle='--')
axes[0].axhline(mean - z_threshold * std, color='red', linestyle='--')
axes[0].set_title('Z-Score Method')
axes[0].set_ylabel('Value')

# IQR
mask_iqr = (data < lower_fence) | (data > upper_fence)
axes[1].scatter(range(len(data)), data, c=mask_iqr, cmap='coolwarm', alpha=0.6)
axes[1].axhline(upper_fence, color='red', linestyle='--')
axes[1].axhline(lower_fence, color='red', linestyle='--')
axes[1].set_title('IQR Method')

# Box plot
axes[2].boxplot(data, vert=True)
axes[2].set_title('Box Plot (IQR visual)')

plt.tight_layout()
plt.show()

In [ ]:
# Compare methods on skewed data
skewed = np.random.exponential(10, 200)
skewed = np.append(skewed, [80, 90, 100])  # Add outliers

# Z-score on skewed data
z = np.abs((skewed - np.mean(skewed)) / np.std(skewed))
# IQR on skewed data
Q1, Q3 = np.percentile(skewed, [25, 75])
iqr_mask = (skewed < Q1 - 1.5*(Q3-Q1)) | (skewed > Q3 + 1.5*(Q3-Q1))

print(f"\nSKEWED DATA COMPARISON:")
print(f"  Z-score outliers: {np.sum(z > 2.5)}")
print(f"  IQR outliers: {np.sum(iqr_mask)}")
print(f"  Note: IQR is more robust for skewed distributions")

## Key Observations

- Z-score assumes normality; works well for symmetric data.
- IQR method is robust to non-normal distributions.
- Modified Z-score (MAD) is resist to outliers in the detection itself.
- Different methods may flag different points as outliers.
- Domain knowledge should guide final outlier decisions.

In [ ]:
# Optional: Try sklearn's IsolationForest
# from sklearn.ensemble import IsolationForest
# clf = IsolationForest(contamination=0.05, random_state=42)
# preds = clf.fit_predict(data.reshape(-1, 1))
# print(f"IsolationForest outliers: {np.sum(preds == -1)}")